In [2]:
import tensorflow as tf
from tensorflow import keras 
from keras.models import Sequential
from keras.layers import Flatten, Dense
from keras.applications import EfficientNetB0

2025-03-11 19:18:22.397434: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-11 19:18:22.407646: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-11 19:18:22.506390: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-11 19:18:22.588835: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741700902.664947    3131 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741700902.68

In [3]:
conv_base = EfficientNetB0(
    include_top = False,
    weights = 'imagenet',
    input_shape = (128,128,3)
)

2025-03-11 19:18:25.526734: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
plant_model = Sequential()
plant_model.add(conv_base)
plant_model.add(Flatten())
plant_model.add(Dense(19, activation='softmax'))

In [5]:
conv_base.trainable = False

In [6]:
plant_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20480)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 19)             │       389,139 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,438,710 (16.93 MB)

 Trainable params: 389,139 (1.48 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [7]:
dataset = keras.preprocessing.image_dataset_from_directory(
    directory='./Dataset/Leaf',
    shuffle=True,
    batch_size = 16,
    image_size = (128,128)
)

Found 65451 files belonging to 19 classes.


In [8]:
class_names = dataset.class_names
print(class_names)

['Apple', 'Bell Pepper', 'Cherry', 'Coffee', 'Corn', 'Cotton', 'Cucumber', 'Grape', 'Guava', 'Lemon', 'Mango', 'Peach', 'Potato', 'Rice', 'Strawberry', 'Sugarcane', 'Tea', 'Tomato', 'Wheat']


In [9]:
dataset.cardinality().numpy()

4091

In [10]:
train_ds = dataset.take(2864)
valid_ds = dataset.skip(2864).take(818)
test_ds = dataset.skip(3682)

In [11]:
print(train_ds.cardinality())
print(valid_ds.cardinality())
print(test_ds.cardinality())

tf.Tensor(2864, shape=(), dtype=int64)
tf.Tensor(818, shape=(), dtype=int64)
tf.Tensor(409, shape=(), dtype=int64)


In [12]:
def preprocess(image, label):
    label = tf.one_hot(label, depth=19)  # Convert labels to one-hot encoding
    return image, label

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().map(preprocess).shuffle(buffer_size=1000).prefetch(buffer_size=AUTOTUNE)
valid_ds = valid_ds.cache().map(preprocess).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().map(preprocess).prefetch(buffer_size=AUTOTUNE)

In [13]:
plant_model.compile(
    optimizer = 'adam',
    metrics = ['accuracy'],
    loss = 'categorical_crossentropy'
)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True)
]

In [15]:
history = plant_model.fit(
    train_ds,
    validation_data = valid_ds,
    epochs = 50,
    callbacks=callbacks
)

Epoch 1/50


Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS

   4/2864 ━━━━━━━━━━━━━━━━━━━━ 10:53 228ms/step - accuracy: 0.2474 - loss: 2.8840   

Invalid SOS parameters for sequential JPEG


  53/2864 ━━━━━━━━━━━━━━━━━━━━ 4:32 97ms/step - accuracy: 0.6509 - loss: 1.5060

...

1859/2864 ━━━━━━━━━━━━━━━━━━━━ 1:35 95ms/step - accuracy: 0.9103 - loss: 0.4217

Invalid SOS parameters for sequential JPEG


2864/2864 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.9232 - loss: 0.3822

Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS

2864/2864 ━━━━━━━━━━━━━━━━━━━━ 436s 144ms/step - accuracy: 0.9232 - loss: 0.3822 - val_accuracy: 0.9642 - val_loss: 0.2811
Epoch 2/50
2864/2864 ━━━━━━━━━━━━━━━━━━━━ 332s 116ms/step - accuracy: 0.9794 - loss: 0.1313 - val_accuracy: 0.9765 - val_loss: 0.2604
Epoch 3/50
2864/2864 ━━━━━━━━━━━━━━━━━━━━ 326s 114ms/step - accuracy: 0.9856 - loss: 0.1060 - val_accuracy: 0.9720 - val_loss: 0.3469
Epoch 4/50
2864/2864 ━━━━━━━━━━━━━━━━━━━━ 326s 114ms/step - accuracy: 0.9889 - loss: 0.0948 - val_accuracy: 0.9782 - val_loss: 0.3575
Epoch 5/50
2864/2864 ━━━━━━━━━━━━━━━━━━━━ 326s 114ms/step - accuracy: 0.9917 - loss: 0.0692 - val_accuracy: 0.9819 - val_loss: 0.3325
Epoch 6/50
2864/2864 ━━━━━━━━━━━━━━━━━━━━ 326s 114ms/step - accuracy: 0.9930 - loss: 0.0654 - val_accuracy: 0.9781 - val_loss: 0.3913
Epoch 7/50
2864/2864 ━━━━━━━━━━━━━━━━━━━━ 327s 114ms/step - accuracy: 0.9940 - loss: 0.0588 - val_accuracy: 0.9774 - val_loss: 0.4645


In [16]:
plant_model.save('../saved_models/plant_model.keras')

In [17]:
plant_model.evaluate(test_ds)

Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS

  1/409 ━━━━━━━━━━━━━━━━━━━━ 7:43:07 68s/step - accuracy: 1.0000 - loss: 0.0000e+00

Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG


 35/409 ━━━━━━━━━━━━━━━━━━━━ 41s 111ms/step - accuracy: 0.9741 - loss: 0.2610

Invalid SOS parameters for sequential JPEG


...

Invalid SOS parameters for sequential JPEG


353/409 ━━━━━━━━━━━━━━━━━━━━ 5s 103ms/step - accuracy: 0.9743 - loss: 0.2629

Invalid SOS parameters for sequential JPEG


409/409 ━━━━━━━━━━━━━━━━━━━━ 109s 101ms/step - accuracy: 0.9748 - loss: 0.2579


[0.23516307771205902, 0.9776723980903625]